# 03 — Regional Data: Multiple Floats

So far we have looked at a single float.  But one of the great strengths of the Argo
programme is the **global array** of thousands of floats.  By fetching all floats within
a geographic box and time window, we can build a regional picture of ocean conditions.

In this notebook you will:
1. Define a region (bounding box + time range) and download all Argo data in it
2. Map the float trajectories
3. Plot T/S (temperature–salinity) diagrams for the region
4. Compare MLD across multiple floats

In [ ]:
# --- Imports ---
from argopy import DataFetcher as ArgoDataFetcher

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.cm as cm
import matplotlib.patches as mpatches

## 1 — Define the region

We specify:
- A **longitude range** (west, east)
- A **latitude range** (south, north)
- A **pressure range** (surface to max depth)
- A **time range** (start date, end date)

> **Warning:** Large regions with long time windows can mean a very large download.
> Start with a short time window (1–3 months) and a small box, then expand.

The default below covers the region between Iceland, the Faero Island, and Norway

In [ ]:
# ---------------------------------------------------------------
# Edit this box to study a different region or time period
# Longitude: −180 to +180 (negative = west)
# Latitude:  −90  to +90  (negative = south)
# ---------------------------------------------------------------
LON_MIN  = -15
LON_MAX  = 10
LAT_MIN  =  60
LAT_MAX  =  70
PRES_MIN =   0
PRES_MAX = 200

DATE_START   = '2025-09'
DATE_END = '2026-01'
# ---------------------------------------------------------------

region = [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX,
          PRES_MIN, PRES_MAX,
          DATE_START, DATE_END]

print('Region defined:')
print(f'  Longitude: {LON_MIN}° to {LON_MAX}°E')
print(f'  Latitude:  {LAT_MIN}° to {LAT_MAX}°N')
print(f'  Pressure:  {PRES_MIN} to {PRES_MAX} dbar')
print(f'  Period:    {DATE_START}  to  {DATE_END}')

## 2 — Download regional data

In [ ]:
print('Downloading regional Argo data ...  (this may take a while...)')

ds = ArgoDataFetcher(src='erddap').region(region).to_xarray()
df = ds.to_dataframe().reset_index()

# Basic cleaning
df = df.dropna(subset=['TEMP', 'PSAL', 'PRES'])

n_floats = df['PLATFORM_NUMBER'].nunique()
n_profiles = df.groupby(['PLATFORM_NUMBER', 'CYCLE_NUMBER']).ngroups

print(f'Done!')
print(f'  Floats found:    {n_floats}')
print(f'  Profiles found:  {n_profiles}')
print(f'  Total rows:      {len(df):,}')

## 3 — Map float positions

Let's plot where each float was during this period.

In [ ]:
# Get one position per profile (use the median lat/lon per cycle)
positions = (
    df.groupby(['PLATFORM_NUMBER', 'CYCLE_NUMBER'])
    .agg(LON=('LONGITUDE', 'median'),
         LAT=('LATITUDE',  'median'),
         TIME=('TIME',     'median'))
    .reset_index()
)

# Assign a colour to each float
floats = positions['PLATFORM_NUMBER'].unique()
colours = cm.tab20(np.linspace(0, 1, len(floats)))
colour_map = dict(zip(floats, colours))

fig, ax = plt.subplots(figsize=(9, 6))

for wmo in floats:
    sub = positions[positions['PLATFORM_NUMBER'] == wmo].sort_values('TIME')
    ax.plot(sub['LON'], sub['LAT'],
            color=colour_map[wmo], linewidth=0.8, alpha=0.7)
    ax.scatter(sub['LON'].iloc[0],  sub['LAT'].iloc[0],
               color='green', s=40, zorder=5)    # start
    ax.scatter(sub['LON'].iloc[-1], sub['LAT'].iloc[-1],
               color='red',   s=40, zorder=5)    # end

# Draw the bounding box
rect = mpatches.FancyArrowPatch(
    (LON_MIN, LAT_MIN), (LON_MAX, LAT_MAX),
    arrowstyle='-', color='black',
    linestyle='--', linewidth=1.5
)
box = plt.Polygon(
    [[LON_MIN, LAT_MIN], [LON_MAX, LAT_MIN],
     [LON_MAX, LAT_MAX], [LON_MIN, LAT_MAX]],
    fill=False, edgecolor='black', linewidth=1.5, linestyle='--'
)
ax.add_patch(box)

# Legend entries
ax.scatter([], [], color='green', s=40, label='Start position')
ax.scatter([], [], color='red',   s=40, label='End position')
ax.legend(fontsize=10, loc='upper right')

ax.set_xlabel('Longitude (°E)', fontsize=12)
ax.set_ylabel('Latitude (°N)',  fontsize=12)
ax.set_title(f'Argo float trajectories\n{DATE_START} to {DATE_END}', fontsize=13)
ax.set_xlim(LON_MIN - 3, LON_MAX + 3)
ax.set_ylim(LAT_MIN - 3, LAT_MAX + 3)
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('regional_map.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'{n_floats} floats plotted.')

## 4 — Temperature–Salinity (T–S) diagram

A T–S diagram plots **temperature on the y-axis** and **salinity on the x-axis**.  
Each point is one measurement.  Water masses from different origins occupy distinct
regions of T–S space, so this plot is a classic tool for identifying water mass origins.

In [ ]:
# Colour the T-S points by pressure (depth) to show where in the water column
# each water mass sits
fig, ax = plt.subplots(figsize=(7, 6))

# Subsample to avoid plotting millions of points — plot every 5th row
sample = df.iloc[::5]

sc = ax.scatter(
    sample['PSAL'], sample['TEMP'],
    c=sample['PRES'],
    cmap='plasma_r',
    s=1, alpha=0.4
)

cbar = fig.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label('Pressure (dbar)', fontsize=11)

ax.set_xlabel('Salinity (PSU)', fontsize=12)
ax.set_ylabel('Temperature (°C)', fontsize=12)
ax.set_title(f'T–S diagram — {n_floats} floats\n{DATE_START} to {DATE_END}', fontsize=13)
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

## 5 — Compare MLD across floats

We calculate MLD for every profile across all floats and then compare their
seasonal cycles.

In [ ]:
def calculate_mld(profile, threshold=0.2, ref_depth=10):
    """
    Temperature threshold MLD (de Boyer Montégut et al. 2004).
    Returns MLD in dbar, or NaN if it cannot be determined.
    """
    profile = profile.sort_values('PRES').reset_index(drop=True)
    near_surface = profile[
        (profile['PRES'] >= ref_depth - 5) &
        (profile['PRES'] <= ref_depth + 5)
    ]
    if near_surface.empty:
        near_surface = profile.iloc[[0]]
    T_ref = near_surface['TEMP'].mean()
    if np.isnan(T_ref):
        return np.nan
    mixed_layer = profile[profile['TEMP'] >= T_ref - threshold]
    return np.nan if mixed_layer.empty else mixed_layer['PRES'].max()


# Compute MLD for every (float, cycle) combination
print('Computing MLD for all profiles ...')

mld_records = []
for (wmo, cycle), group in df.groupby(['PLATFORM_NUMBER', 'CYCLE_NUMBER']):
    mld = calculate_mld(group)
    mld_records.append({
        'WMO':   wmo,
        'CYCLE': cycle,
        'TIME':  group['TIME'].median(),
        'LAT':   group['LATITUDE'].median(),
        'LON':   group['LONGITUDE'].median(),
        'MLD':   mld
    })

mld_all = pd.DataFrame(mld_records).dropna(subset=['MLD'])
print(f'Done!  {len(mld_all)} valid MLD estimates across {mld_all["WMO"].nunique()} floats.')

In [ ]:
# --- Plot individual float MLD time series ---
fig, ax = plt.subplots(figsize=(12, 5))

floats_with_data = mld_all['WMO'].unique()

# Limit to 10 floats to keep the plot readable
for wmo in floats_with_data[:10]:
    sub = mld_all[mld_all['WMO'] == wmo].sort_values('TIME')
    ax.plot(sub['TIME'], sub['MLD'],
            color=colour_map[wmo], linewidth=1.2, alpha=0.8,
            label=str(wmo))

ax.invert_yaxis()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))

ax.set_ylabel('Mixed Layer Depth (dbar)', fontsize=12)
ax.set_title(f'MLD — {n_floats} Argo floats  ({DATE_START} to {DATE_END})', fontsize=13)
ax.legend(title='Float WMO', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('regional_mld.png', dpi=150, bbox_inches='tight')
plt.show()

## 6 — Spatial map of mean MLD

Plot the time-averaged MLD at each float's mean position — a simple spatial map.

In [ ]:
# Average MLD and position for each float
float_summary = (
    mld_all.groupby('WMO')
    .agg(MEAN_MLD=('MLD',  'mean'),
         MEAN_LAT=('LAT', 'mean'),
         MEAN_LON=('LON', 'mean'))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 6))

sc = ax.scatter(
    float_summary['MEAN_LON'],
    float_summary['MEAN_LAT'],
    c=float_summary['MEAN_MLD'],
    cmap='Blues',       # Deeper MLD = darker blue
    s=120,
    edgecolors='black', linewidth=0.5,
    zorder=5
)

cbar = fig.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label('Mean MLD (dbar)', fontsize=11)

# Label each dot with the WMO number
for _, row in float_summary.iterrows():
    ax.annotate(str(int(row['WMO'])),
                xy=(row['MEAN_LON'], row['MEAN_LAT']),
                xytext=(4, 4), textcoords='offset points',
                fontsize=7)

ax.set_xlim(LON_MIN - 2, LON_MAX + 2)
ax.set_ylim(LAT_MIN - 2, LAT_MAX + 2)
ax.set_xlabel('Longitude (°E)', fontsize=12)
ax.set_ylabel('Latitude (°N)',  fontsize=12)
ax.set_title(f'Mean MLD per float ({DATE_START} to {DATE_END})', fontsize=13)
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('regional_mld_map.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Exercises

1. **Change the region** — Try the Mediterranean Sea:
   `LON_MIN=-6, LON_MAX=36, LAT_MIN=30, LAT_MAX=46`.
   How does the T–S diagram differ from the North Atlantic?

2. **Longer time window** — Extend `DATE_START` to `'2025-01'` and see how the seasonal
   MLD cycle repeats.